# Notebook 2: Exploring TB and Socio-economic Indicators, Part 1

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/NB02-World-TB-Notebook-Part-1.ipynb)



Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `NB02-World-TB-Notebook-Part-1.ipynb` at `6d50f558cd39b40089cd3eea1fbc19425e5191d1`; original SHA-256 `413ae8aa277181215c434552d8a2dfe3df7f69442c3de31af58621785ceaddea`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run ./resources/library.py

In [ ]:
style_notebook()

# Notebook 2: Exploring TB and Socio-economic Indicators, Part 1

## Goals

Our goal for this TB exploration notebook is to construct a "gapminder" for TB data and a time series choropleth map. See figures below.

[Source image continuation: Gapminder for TB](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/images/gapminder-for-tb.png)

**Figure 1**. Interactive Gapminder bubble chart for TB data

[Source image continuation: Choropleth Timeseries for TB](https://github.com/PHI-Case-Studies/2005-MDR-TB-Outbreak-Thailand/blob/6d50f558cd39b40089cd3eea1fbc19425e5191d1/images/choropleth-timeseries-tb.png)

**Figure 2**. Interactive choropleth map for TB data

## Steps

Notebook 2
1. Set up data exploration environment
2. Download and clean up data from the WHO, World Bank

## STEP 1. Set up data exploration environment

To create visualizations, we will use Plotly Express. You can learn more about Plotly Express from:
1. [Plotly GitHub Site](https://github.com/plotly/plotly_express) (GitHub)
2. [Jupyter Notebook Example](https://www.plotly.express/)
3. Some articles written about it [here](https://medium.com/@plotlygraphs/introducing-plotly-express-808df010143d) and [here](https://towardsdatascience.com/plotly-express-the-good-the-bad-and-the-ugly-dc941649687c)

In [ ]:
import plotly.express as px
import plotly

plotly.__version__

We will also use our Swiss Army knife for data science, `pandas`.

In [ ]:
import pandas as pd
pd.__version__

In [ ]:
pd.set_option('max_colwidth', 150)
pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 1000)

**`missingno`**: Missing data visualization module for Python 

Read more about it here: https://github.com/ResidentMario/missingno

In [ ]:
import missingno as mno
mno.__version__

In [ ]:
import matplotlib

matplotlib.__version__

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np

np.__version__

## STEP 2. Download and clean up data

All data files are found in the `resources` folder. You can also download these using the links and instructions below.

### 1. World Health Organization (WHO), estimates of TB burden

Data from 2000-2017 provided by countries to WHO and estimates of TB burden generated by WHO for the Global Tuberculosis Report. 

1. Source, data: https://extranet.who.int/tme/generateCSV.asp?ds=estimates
2. Source, metadata: https://extranet.who.int/tme/generateCSV.asp?ds=dictionary
3. Instructions: No need to modify CSV download.

**Aggregate Data**

In [ ]:
tb_df1 = pd.read_csv("resources/TB_burden_countries_2019-07-03.csv")

tb_df1.head()

In [ ]:
tb_df1.columns

In [ ]:
mno.bar(tb_df1)

**Data Dictionary**

You can look up the definitions of the columns above using the data dictionary dataframe.

In [ ]:
tb_dict_df = pd.read_csv("resources/TB_data_dictionary_2019-07-03.csv")

tb_dict_df.head()

You can also query the dataframe for specific variables, as shown below.

In [ ]:
tb_dict_df.query("variable_name=='e_mort_num'")

In [ ]:
tb_dict_df.query("variable_name=='e_inc_100k'")

In [ ]:
tb_dict_df.query("variable_name=='e_inc_tbhiv_100k'")

#### Prepare for second merge between TB data and the merged WB data

In [ ]:
tb_df1.head()

For ease of merges between dataframes, we will use common field names:
1. `Country` (country name)
2. `ISO_Alpha` (3-letter ISO country code)
3. `Year`

In [ ]:
tb_df2 = tb_df1.rename(\
        columns={'country': 'Country', 'iso3': 'ISO_Alpha', 'year': 'Year'})

tb_df2.head()

#### Pickle WHO TB data set

In [ ]:
tb_df2.to_pickle('outputs/tb_df2.pickle')
tb_dict_df.to_pickle('outputs/tb_dict_df.pickle')

### 2. World Bank (WB), World Gross Domestic Product (GDP) Data, 1960-2018

**Aggregate Data, GDP**

1. Source: http://api.worldbank.org/v2/en/indicator/NY.GDP.MKTP.CD?downloadformat=csv
2. License: https://datacatalog.worldbank.org/public-licenses#cc-by
3. Note: The downloaded zip file has 3 CSV files: data, metadata and indicator metatadata (first two are used here). The data CSV file has 4 rows at the top that should be deleted.

In [ ]:
wb_gdp_df1 = pd.read_csv("resources/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_123-V2.csv")

wb_gdp_df1.head()

Because this data set is just about one indicator, GDP, let's drop the two columns, `Indicator Name` and `Indicator Code` and rename `Country Name` to `Country` and `Country Code` to `ISO_Alpha`.

Python indices start with zero), hence:
1. Column 0: "Country Name"
2. Column 1: "Country Code"
3. Column 2: "Indicator Name"
4. Column 3: "Indicator Code"

In [ ]:
# columns to be dropped, as above
cols = [2,3]

# drop columns 2 and 3 and rename columns
wb_gdp_df2 = wb_gdp_df1.drop(wb_gdp_df1.columns[cols],axis=1)
wb_gdp_df3 = wb_gdp_df2.rename(columns={ "Country Name": "Country", \
                                        "Country Code": "ISO_Alpha" })
wb_gdp_df3

#### Aggregate data, GDP per capita

1. Source: http://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.CD?downloadformat=csv (version: 7/10/2019)
2. License: https://datacatalog.worldbank.org/public-licenses#cc-by
3. Note: The downloaded zip file has 3 CSV files: data, metadata and indicator metatadata (only one used here, metadata file is similar to GDP). The data CSV file has 4 rows at the top that should be deleted.

In [ ]:
wb_gdppc_df1 = \
    pd.read_csv("resources/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_41089 V2.csv")

wb_gdppc_df1.head()

In [ ]:
mno.bar(wb_gdppc_df1)

In [ ]:
cols = [2,3]
wb_gdppc_df2 = wb_gdppc_df1.drop(wb_gdppc_df1.columns[cols], axis=1)
wb_gdppc_df3 = wb_gdppc_df2.rename(columns={ "Country Name": "Country", \
                                        "Country Code": "ISO_Alpha" })
wb_gdppc_df3

### Pandas `melt`, prepare WB GDP data for merge

#### GDP data

Pandas melt normalizes the dataframe to `Year`, `Country` and `value`, the latter representing `GDP` or `GDP per capita`in US dollars.

From the `pandas melt` [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.melt.html):
> *Unpivot a DataFrame from wide format to long format, optionally leaving identifier variables set.*
>  
> *This function is useful to massage a DataFrame into a format where one or more columns are identifier variables (id_vars), while all other columns, considered measured variables (value_vars), are “unpivoted” to the row axis, leaving just two non-identifier columns, ‘variable’ and ‘value’.*

Executed as code below, from wide format to long format:

In [ ]:
wb_gdp_df4 = wb_gdp_df3.melt(id_vars=['Country', 'ISO_Alpha'])

wb_gdp_df4.head()

Let's rename `variable` to `Year` and `value` to `GDP-USD`.

In [ ]:
wb_gdp_df5 = wb_gdp_df4.rename(columns={'variable': 'Year', 'value': 'GDP-USD'})

wb_gdp_df5.head()

In [ ]:
wb_gdp_df5.columns

In [ ]:
wb_gdp_df5.dtypes

In [ ]:
wb_gdp_df5['Year'] = pd.to_numeric(wb_gdp_df5['Year'])

In [ ]:
wb_gdp_df5.dtypes

In [ ]:
wb_gdp_df5_2017 = wb_gdp_df5.query("Year==2017")

wb_gdp_df5_2017.head()

In [ ]:
mno.bar(wb_gdp_df5)

#### GDP per capita data

In [ ]:
wb_gdppc_df4 = wb_gdppc_df3.melt(id_vars=['Country', 'ISO_Alpha'])

wb_gdppc_df4.head()

In [ ]:
wb_gdppc_df5 = wb_gdppc_df4.rename(columns={'variable': 'Year', 'value': 'GDP-PC-USD'})

wb_gdppc_df5.head()

In [ ]:
wb_gdppc_df5['Year'] = pd.to_numeric(wb_gdppc_df5['Year'])

In [ ]:
wb_gdppc_df5_2017 = wb_gdppc_df5.query("Year==2017")

wb_gdppc_df5_2017.head()

In [ ]:
mno.bar(wb_gdppc_df5)

### World Bank metadata updates

In [ ]:
wb_gdp_meta_df1 = pd.read_csv("resources/Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_123.csv")

wb_gdp_meta_df1.head()

Let's rename some columns in the WB dataframe above in preparation for dataframe merge. Let's also drop the `Unnamed: 5` and  `SpecialNotes` columns.

In [ ]:
wb_gdp_meta_df2 = wb_gdp_meta_df1.rename(\
        columns={'TableName': 'Country', 'Country Code': 'ISO_Alpha'})
wb_gdp_meta_df3 = wb_gdp_meta_df2.drop(['Unnamed: 5', 'SpecialNotes'], axis=1)

wb_gdp_meta_df3.head()

### Check linkage column, `ISO_Alpha`, before Merge #1

Because we are going to merge WB data and metadata on the `ISO_Alpha` column, we have to check if the data from the `ISO_Alpha` columns from both dataframes match.

To do this we will construct two dataframes that we can visually inspect and compare: `wb_gdp_c1` and `wb_gdp_c2`.

In [ ]:
wb_gdp_iso3 = wb_gdp_df5.ISO_Alpha.unique()
wb_gdp_country = wb_gdp_df5.Country.unique()

wb_gdp_c1 = pd.DataFrame({'ISO_Alpha':wb_gdp_iso3,'Country':wb_gdp_country})

wb_gdp_c1

In [ ]:
wb_gdp_meta_iso3 = wb_gdp_meta_df3.ISO_Alpha.unique()
wb_gdp_meta_country = wb_gdp_meta_df3.Country.unique()

wb_gdp_c2 = pd.DataFrame({'ISO_Alpha':wb_gdp_meta_iso3,'Country':wb_gdp_meta_country})

wb_gdp_c2

Using a `numpy` dot function called `setdiff1d()`, let's check which `Country` names `ISO_Alpha` codes don't overlap between WB GDP data and metadata.

In [ ]:
wb_nonmatch = np.setdiff1d(wb_gdp_iso3, wb_gdp_meta_iso3)

wb_nonmatch

If you check the value `INX`, it does not represent a country.

#### Pickle WB dataframes

In [ ]:
wb_gdp_df5.to_pickle('outputs/wb_gdp_df5.pickle')
wb_gdppc_df5.to_pickle('outputs/wb_gdppc_df5.pickle')
wb_gdp_meta_df3.to_pickle('outputs/wb_gdp_meta_df3.pickle')

### 3. WB Education Data, 1960-2018

From the World Bank education page (link below):

> *"Education is one of the most powerful instruments for reducing poverty and inequality and lays a foundation for sustained economic growth. The World Bank compiles data on education inputs, participation, efficiency, and outcomes. Data on education are compiled by the United Nations Educational, Scientific, and Cultural Organization (UNESCO) Institute for Statistics from official responses to surveys and from reports provided by education authorities in each country."*

1. Source: http://api.worldbank.org/v2/en/topic/4?downloadformat=csv (version: 7/10/2019)
2. License: https://datacatalog.worldbank.org/public-licenses#cc-by
3. Note: The downloaded zip file has 3 CSV files: data, metadata and indicator metatadata (first two are used here). The data CSV file has 4 rows at the top that should be deleted. The metadata file has complete descriptions of indicators, including data sources (see below).

### Aggregate data, WB education indicators

In [ ]:
wb_ed_df1 = pd.read_csv('resources/API_4_DS2_en_csv_v2_48246 V2.csv')

wb_ed_df1.head(30)

### Indicator metadata, WB education indicators

In [ ]:
wb_edmeta_df1 = \
  pd.read_csv('resources/Metadata_Indicator_API_4_DS2_en_csv_v2_48246.csv').\
    drop(['Unnamed: 4'], axis=1)

wb_edmeta_df1.head(160)

Let's rename some columns of the education data set. Note that this data set has several indicators referenced in the metadata data above.

In [ ]:
wb_ed_df2 = wb_ed_df1.rename(columns={ "Country Name": "Country", \
                                        "Country Code": "ISO_Alpha", \
                                       "Indicator Name": "Indicator_Name", \
                                       "Indicator Code": "Indicator_Code"})
wb_ed_df2.head(30)

### Education data subsets

The education data set contains many indicators. We will pick one indicator to merge with other GDP indicators.

#### Primary completion rate

Let's query the metadata for a full description of the primary completion rate indicator. (Note: You can apply the same steps to other education indicators found in the data set.

In [ ]:
# first line disables the column width default so we can see
#   the full text of `SOURCE NOTES` column
# the second line prints the row corresponding to the indicator code
#   for primary completion rate
with pd.option_context('display.max_colwidth', None): 
    print(wb_edmeta_df1.query("INDICATOR_CODE=='SE.PRM.CMPT.ZS'"))

In [ ]:
wb_ed_primcomp_df1 = wb_ed_df2.query("Indicator_Code=='SE.PRM.CMPT.ZS'")

wb_ed_primcomp_df1

Let's use `missingo` with alias `mno` to inspect missing values for this indicator across years of data collection.

In [ ]:
mno.bar(wb_ed_primcomp_df1)

This data set being a subset for primary completion rate indicator, we will not need the Indicator_Code and Indicator_Name columns so we drop these using the code below.

In [ ]:
cols = [2,3]
wb_ed_primcomp_df2 = wb_ed_primcomp_df1.drop(wb_ed_primcomp_df1.columns[cols],axis=1)

wb_ed_primcomp_df2

### Pandas `melt`, prepare WB education data for merge

#### Primary completion rate

In [ ]:
wb_ed_primcomp_df3 = wb_ed_primcomp_df2.melt(\
            id_vars=['Country', 'ISO_Alpha'])

wb_ed_primcomp_df3.head(200)

In [ ]:
wb_ed_primcomp_df4 = \
    wb_ed_primcomp_df3.rename(columns={'variable': 'Year', 'value':'Value'})
wb_ed_primcomp_df4['Year'] = pd.to_numeric(wb_ed_primcomp_df4['Year'])

wb_ed_primcomp_df4.head()

### Visually explore "pivoted" dataframes

#### Primary education completion rate

From the `pandas pivot` [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.pivot.html):
> *Return reshaped DataFrame organized by given index / column values.*
>  
> *Reshape data (produce a “pivot” table) based on column values. Uses unique values from specified index / columns to form axes of the resulting DataFrame. This function does not support data aggregation, multiple values will result in a MultiIndex in the columns. See the User Guide for more on reshaping.*

Executed as code below, from long format to wide format:

In [ ]:
wb_ed_primcomp_pivot_df4 = \
    wb_ed_primcomp_df4.pivot(index='Year', columns='Country', values='Value')

wb_ed_primcomp_pivot_df4.head()

In [ ]:
ax = wb_ed_primcomp_pivot_df4.plot(\
    legend=False, \
    color=['black'], \
    figsize=(7,7),\
    title='Primary completion rate, total (% of relevant age group))')
ax.set_xlabel("Year")
ax.set_ylabel("Completion Rate")

Let's look at those countries with completion rate > 125.

In [ ]:
wb_ed_primcomp_df4.query("Value>125")

#### Pickle WB education data

In [ ]:
wb_ed_df2.to_pickle('outputs/wb_ed_df2.pickle')
wb_edmeta_df1.to_pickle('outputs/wb_edmeta_df1.pickle')
wb_ed_primcomp_df4.to_pickle('outputs/wb_ed_primcomp_df4.pickle')

### 4. WB data on health care per capita spending, 1960-2018

1. Source: http://api.worldbank.org/v2/en/indicator/SH.XPD.CHEX.PC.CD?downloadformat=csv (version: 7/10/2019)
2. License: https://datacatalog.worldbank.org/public-licenses#cc-by
3. Note: The downloaded zip file has 3 CSV files: data, metadata and indicator metatadata (first two are used here). The data CSV file has 4 rows at the top that should be deleted. You can further explore which columns actually have data (columns for earlier years might be empty for all countries).

#### Aggregate data, health care per capita spending

In [ ]:
wb_hcpcs_df1 = pd.read_csv('resources/API_SH.XPD.CHEX.PC.CD_DS2_en_csv_v2_41516 V2.csv')

wb_hcpcs_df1.head()

In [ ]:
mno.bar(wb_hcpcs_df1)

Because this data set is just about one indicator, health care per capita spending, let's drop the two columns, `Indicator Name` and `Indicator Code`, drop the columns from `1960` to `1999` because they are empty, and rename `Country Name` to `Country` and `Country Code` to `ISO_Alpha`.

In [ ]:
wb_hcpcs_df2 = wb_hcpcs_df1.drop(wb_hcpcs_df1.iloc[:, 2:44], axis=1)

wb_hcpcs_df2.head()

In [ ]:
wb_hcpcs_df3 = wb_hcpcs_df2.iloc[:, :-2]

wb_hcpcs_df3.head()

In [ ]:
wb_hcpcs_df4 = wb_hcpcs_df3.rename(columns={ "Country Name": "Country", \
                                        "Country Code": "ISO_Alpha" })
wb_hcpcs_df4.head()

#### Pickle WB HCPCS data

In [ ]:
wb_hcpcs_df4.to_pickle('outputs/wb_hcpcs_df4.pickle')

## You finished Notebook 2!

Please proceed to Notebook 3.

## Note

Although you finished Notebook 2, where all data sets are loaded and processed, the World Bank data catalog has many more data sets that could be explored, related to poverty, climate, infrastructure, and others.